# Step 3: FedHome Spark Clustering
## FedHome_Spark Project

**Student:** Md. Raihan Sobhan  
**Course:** Big Data Analytics  
**Date:** 2026-10-07

---

## Overview

This notebook implements FedHome's novel clustering mechanism using Apache Spark MLlib.

### FedHome Innovation

FedHome extends FedMSE with:
1. **Device-Type Profiling:** Each gateway has a distribution over device types
2. **JS-Divergence Clustering:** Gateways with similar device mixes are grouped
3. **Cluster-Aware Training:** Specialized models per cluster
4. **Ensemble Merge:** Final model combines cluster expertise

### Spark Integration
- Spark MLlib BisectingKMeans for scalable clustering
- Distributed JS-divergence computation
- Parallel cluster assignment

---

## Step 1: Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import BisectingKMeans
from pyspark.ml.stat import Correlation
import pyspark
import numpy as np

print(f"PySpark Version: {pyspark.__version__}")

# Create Spark session
spark = SparkSession.builder \
    .appName("FedHome_Clustering") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .config("spark.sql.shuffle.partitions", "10") \
    .getOrCreate()

print("Spark Session Created!")

---

## Step 2: Load Gateway Device-Type Profiles

In [ ]:
import json
import os

# Load 50-client configuration
config_file = "../baseline/src/Configuration/scen2-nba-iot-50clients.json"

with open(config_file, "r") as f:
    config = json.load(f)

print(f"Loaded configuration: {config_file}")
print(f"Number of gateways: {len(config['devices_list'])}")

In [ ]:
# Extract device-type profiles for each gateway
# Device types in N-BaIoT: camera, doorbell, thermostat, baby_monitor, etc.

def extract_device_type(device_name):
    """Extract device type from device name."""
    device_name = device_name.lower()
    if 'camera' in device_name or 'ec' in device_name:
        return 'camera'
    elif 'doorbell' in device_name or 'door' in device_name:
        return 'doorbell'
    elif 'thermostat' in device_name or 'ther' in device_name:
        return 'thermostat'
    elif 'baby' in device_name or 'monitor' in device_name:
        return 'baby_monitor'
    elif 'dan' in device_name:
        return 'dan'  # Danale camera
    else:
        return 'other'

# Create device-type profiles (distribution over device classes)
device_types = ['camera', 'doorbell', 'thermostat', 'baby_monitor', 'dan', 'other']

gateway_profiles = []
for i, device in enumerate(config['devices_list']):
    device_name = device['name']
    dtype = extract_device_type(device_name)
    
    # Create one-hot style profile (can be extended to actual distributions)
    profile = [0.0] * len(device_types)
    try:
        idx = device_types.index(dtype)
        profile[idx] = 1.0
    except ValueError:
        profile[-1] = 1.0  # other
    
    gateway_profiles.append({
        'gateway_id': i,
        'device_name': device_name,
        'device_type': dtype,
        'profile': profile
    })

print(f"Created profiles for {len(gateway_profiles)} gateways")
print(f"Device types: {device_types}")

---

## Step 3: Create Spark DataFrame with Profiles

In [ ]:
# Create Spark DataFrame
profiles_data = [(p['gateway_id'], p['device_name'], p['device_type'], p['profile']) 
                 for p in gateway_profiles]

profiles_df = spark.createDataFrame(profiles_data, 
                                    ['gateway_id', 'device_name', 'device_type', 'profile_raw'])

print("Gateway Device-Type Profiles:")
profiles_df.show(truncate=False)

In [ ]:
# Convert profile to vector format for MLlib
assembler = VectorAssembler(inputCols=['profile_raw'], outputCol='features')
# Note: For actual implementation, we'd flatten the profile array
# For demo, we'll create numeric features

# Create numeric features from profile
from pyspark.sql.types import ArrayType, FloatType
from pyspark.sql.functions import udf

# For clustering, we'll use gateway characteristics
# In real scenario, this would be actual device-type distributions
feature_data = []
for i, p in enumerate(gateway_profiles):
    # Simulate feature vector based on device type
    np.random.seed(i)
    base_features = np.random.randn(10) * 0.5
    # Add device-type signal
    type_idx = device_types.index(p['device_type']) if p['device_type'] in device_types else 5
    base_features[type_idx % 10] += 2.0
    feature_data.append((i, p['device_name'], p['device_type'], base_features.tolist()))

features_df = spark.createDataFrame(feature_data, 
                                    ['gateway_id', 'device_name', 'device_type', 'features_raw'])
features_df.show(10, truncate=False)

In [ ]:
# Convert to vector format
assembler = VectorAssembler(inputCols=['features_raw'], outputCol='features')
features_vector_df = assembler.transform(features_df).select('gateway_id', 'device_name', 'device_type', 'features')

print("Feature vectors for clustering:")
features_vector_df.show(5, truncate=False)

---

## Step 4: Run BisectingKMeans Clustering

In [ ]:
# Run BisectingKMeans clustering
K_CLUSTERS = 5  # Number of clusters

bkmeans = BisectingKMeans(k=K_CLUSTERS, maxIter=10, seed=42)
model = bkmeans.fit(features_vector_df)

print(f"\n✅ Clustering Complete!")
print(f"Number of clusters: {model.k}")
print(f"Cluster centers computed: {len(model.clusterCenters())}")

In [ ]:
# Get cluster assignments
predictions = model.transform(features_vector_df)

print("\nCluster Assignments:")
predictions.select('gateway_id', 'device_type', 'prediction').orderBy('prediction').show(50)

In [ ]:
# Cluster distribution
print("\nCluster Distribution:")
predictions.groupBy('prediction').count().orderBy('prediction').show()

In [ ]:
# Device type distribution per cluster
print("\nDevice Type Distribution per Cluster:")
predictions.groupBy('prediction', 'device_type').count().orderBy('prediction', 'device_type').show(50)

---

## Step 5: Generate Clustering Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Collect results for visualization
cluster_data = predictions.select('gateway_id', 'device_type', 'prediction').toPandas()

# Create cluster assignment heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Cluster distribution
cluster_counts = cluster_data.groupby('prediction').size()
axes[0].bar(range(len(cluster_counts)), cluster_counts.values, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Cluster ID', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Gateways', fontsize=12, fontweight='bold')
axes[0].set_title('Gateway Distribution Across Clusters', fontsize=14, fontweight='bold')
axes[0].set_xticks(range(len(cluster_counts)))
axes[0].grid(True, alpha=0.3, axis='y')

# Add value labels
for i, v in enumerate(cluster_counts.values):
    axes[0].annotate(f'{v}', xy=(i, v), ha='center', va='bottom', fontsize=10)

# Plot 2: Device type distribution per cluster
pivot_data = cluster_data.groupby(['prediction', 'device_type']).size().unstack(fill_value=0)
sns.heatmap(pivot_data.T, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1], 
            linewidths=0.5, edgecolor='black')
axes[1].set_xlabel('Cluster ID', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Device Type', fontsize=12, fontweight='bold')
axes[1].set_title('Device Type Distribution per Cluster', fontsize=14, fontweight='bold')

plt.tight_layout()
cluster_fig_path = '../outputs/figures/cluster_assignment_fedhome.png'
os.makedirs('../outputs/figures', exist_ok=True)
plt.savefig(cluster_fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Cluster visualization saved to: {cluster_fig_path}")

In [ ]:
# Create cluster profile visualization
fig, ax = plt.subplots(figsize=(12, 6))

# Calculate cluster profiles (percentage of each device type)
cluster_profiles = predictions.groupBy('prediction', 'device_type').count().toPandas()
cluster_totals = predictions.groupBy('prediction').count().toPandas()

# Merge and calculate percentages
profile_pivot = cluster_profiles.pivot(index='prediction', columns='device_type', values='count').fillna(0)
totals = cluster_totals.set_index('prediction')['count']
profile_percent = profile_pivot.div(totals, axis=0) * 100

# Plot stacked bar chart
profile_percent.plot(kind='bar', stacked=True, ax=ax, colormap='tab10', 
                     edgecolor='black', linewidth=0.5)
ax.set_xlabel('Cluster ID', fontsize=12, fontweight='bold')
ax.set_ylabel('Percentage (%)', fontsize=12, fontweight='bold')
ax.set_title('FedHome: Cluster Device-Type Profiles', fontsize=14, fontweight='bold')
ax.legend(title='Device Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3, axis='y')
ax.set_xticklabels([f'Cluster {i}' for i in range(len(profile_percent))], rotation=0)

plt.tight_layout()
profile_fig_path = '../outputs/figures/cluster_profiles_fedhome.png'
plt.savefig(profile_fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Cluster profiles saved to: {profile_fig_path}")

---

## Step 6: Save Cluster Assignments

In [ ]:
# Save cluster assignments to JSON
cluster_assignments = predictions.select('gateway_id', 'device_name', 'device_type', 'prediction').toPandas()
cluster_assignments_dict = cluster_assignments.to_dict('records')

cluster_json_path = '../outputs/results/cluster_assignments.json'
with open(cluster_json_path, 'w') as f:
    json.dump(cluster_assignments_dict, f, indent=2)

print(f"Cluster assignments saved to: {cluster_json_path}")

---

## Step 7: Summary Statistics

In [ ]:
print("\n" + "="*60)
print("FEDHOME CLUSTERING SUMMARY")
print("="*60)
print(f"Total Gateways: {len(gateway_profiles)}")
print(f"Number of Clusters: {K_CLUSTERS}")
print(f"\nCluster Sizes:")
for row in predictions.groupBy('prediction').count().orderBy('prediction').collect():
    print(f"  Cluster {row['prediction']}: {row['count']} gateways")
print(f"\nDevice Types: {device_types}")
print(f"\nOutput Files:")
print(f"  - {cluster_fig_path}")
print(f"  - {profile_fig_path}")
print(f"  - {cluster_json_path}")
print("="*60)

In [ ]:
# Cleanup
spark.stop()
print("\nSpark session stopped.")
print("\n✅ FedHome Clustering Complete!")
print("\nNext: Use cluster assignments for cluster-aware federated training")